In [ ]:
import pandas as pd, numpy as np, re, glob, os
from google.colab import files
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)

# Upload only if the files aren't already in the Colab session
if not (glob.glob("*ahindra*.xlsx") and glob.glob("*ajaj*.xlsx")):
    files.upload()            # pick BOTH Excel files in the dialog and wait for "Saving ... 100% done"

def find(pattern):
    hits = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not hits:
        raise FileNotFoundError(f"No file matching {pattern} in {os.getcwd()}. Files here: {os.listdir('.')}")
    return hits[-1]           # newest match, in case of duplicates like "(1)"

MAH_PATH = find("*ahindra*.xlsx")
BAJ_PATH = find("*ajaj*.xlsx")
print("Using:", MAH_PATH, "|", BAJ_PATH)

FY_RE = r"^FY\d{4}-\d{2}$"

def to_num(s):
    """'1,15,159' / 'n/a' / '-' / blanks -> float or NaN"""
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")

def read_baj(sheet, mapping, header_row=4):
    df = pd.read_excel(BAJ_PATH, sheet_name=sheet, header=header_row - 1)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={df.columns[0]: "FY"})
    df = df[df["FY"].astype(str).str.match(FY_RE)]          # drops notes / blank rows
    df = df[["FY"] + list(mapping)].rename(columns=mapping)
    for c in df.columns[1:]:
        df[c] = to_num(df[c])
    return df.set_index("FY")

Saving Bajaj_Finance_10yr_Dataset.xlsx to Bajaj_Finance_10yr_Dataset.xlsx
Saving Mahindra Data.xlsx to Mahindra Data.xlsx
Using: Mahindra Data.xlsx | Bajaj_Finance_10yr_Dataset.xlsx


In [ ]:
# ---------- Mahindra (Annual_Data) ----------
mah = pd.read_excel(MAH_PATH, sheet_name="Annual_Data")
mah.columns = mah.columns.str.strip()
mah = mah[mah["FY"].astype(str).str.match(FY_RE)].copy()
for c in mah.columns.difference(["FY", "Company", "Reporting_Basis"]):
    mah[c] = to_num(mah[c])

mah = mah.rename(columns={
    "Reporting_Basis": "Basis", "AUM_INR_Cr": "AUM", "Gross_NPA_INR_Cr": "Gross_NPA", "GNPA_Pct_Reported": "GNPA_rep",
    "Net_NPA_INR_Cr": "Net_NPA", "NNPA_Pct_Reported": "NNPA_rep", "Stage1_Assets_INR_Cr": "Stage1",
    "Stage2_Assets_INR_Cr": "Stage2", "Stage3_Assets_INR_Cr": "Stage3", "Credit_Cost_Pct_Reported": "CreditCost_rep",
    "ROA_Pct_Reported": "ROA_rep", "CRAR_Pct": "CRAR", "Tier1_Capital_Pct": "Tier1",
    "Gross_Loans_Advances_INR_Cr": "Gross_Loans", "Total_Assets_Closing_INR_Cr": "Total_Assets",
    "Net_Worth_Closing_INR_Cr": "Net_Worth", "Total_Borrowings_INR_Cr": "Borrowings",
    "Total_Income_INR_Cr": "Total_Income", "Interest_Expense_INR_Cr": "Interest_Exp", "Net_Total_Income_INR_Cr": "NTI",
    "Opex_INR_Cr": "Opex", "Impairment_Expense_INR_Cr": "Impairment", "PBT_INR_Cr": "PBT", "PAT_INR_Cr": "PAT",
    "Writeoffs_Annual_INR_Cr": "Writeoffs", "Stage1_ECL_Provision_INR_Cr": "ECL_S1", "Stage2_ECL_Provision_INR_Cr": "ECL_S2",
    "Stage3_ECL_Provision_INR_Cr": "ECL_S3", "Cost_of_Funds_Pct": "CoF", "LCR_Pct": "LCR",
    "Cash_and_Bank_Balances_INR_Cr": "Cash_Bank", "Liquidity_Buffer_Reported_INR_Cr": "Liq_Buffer",
})
# ECL_Provision_INR_Cr is only Stage-3 ECL and conflicts with ECL_S3 in FY2020-21 (3734 vs 2360).
# ECL_S3 reconciles with Gross NPA - Net NPA, so keep that. Old liquidity column is superseded by the reported one.
mah = mah.rename(columns={"Liquidity_Buffer_INR_Cr": "Liq_Buffer_Supplied"})
mah = mah.drop(columns=["ECL_Provision_INR_Cr"])
# ---------- Bajaj (topic sheets -> one table) ----------
parts = [
    read_baj("AUM", {"Consolidated AUM": "AUM", "Standalone AUM": "AUM_Standalone", "Assets under finance (AUF)": "AUF"}),
    read_baj("Asset_Quality", {"GNPA % (reported)": "GNPA_rep", "NNPA % (reported)": "NNPA_rep",
                               "Gross NPA ₹ (Stage 3)": "Gross_NPA", "Net NPA ₹": "Net_NPA", "PCR on Stage 3 % (reported)": "PCR_rep"}),
    read_baj("Stage_ECL", {"Stage 1 gross": "Stage1", "Stage 2 gross": "Stage2", "Stage 3 gross": "Stage3",
                           "ECL Stage 1": "ECL_S1", "ECL Stage 2": "ECL_S2", "ECL Stage 3": "ECL_S3", "Total gross loans": "Gross_Loans"}),
    read_baj("Credit_Losses", {"Loan losses & provisions (impairment)": "Impairment", "Credit cost % (reported: loss / avg AUF)": "CreditCost_rep",
                               "Write-offs": "Writeoffs", "One-time / overlay provisions in year": "OneTime_Prov"}),
    read_baj("Capital", {"CRAR %": "CRAR", "Tier 1 %": "Tier1", "Leverage (x, reported)": "Leverage_rep", "Total assets": "Total_Assets",
                         "Shareholders' equity": "Net_Worth", "Borrowings (debt + deposits + sub-debt)": "Borrowings"}),
    read_baj("Profitability", {"Total income": "Total_Income", "Interest expenses": "Interest_Exp", "Net total income (NTI)": "NTI",
                               "Net interest income (NII)": "NII", "Operating expenses": "Opex", "Profit before tax": "PBT",
                               "Profit after tax": "PAT", "ROA % (reported)": "ROA_rep", "ROE % (reported)": "ROE_rep", "Cost of funds %": "CoF"}),
    read_baj("Liquidity", {"Liquidity buffer": "Liq_Buffer", "Liquidity coverage ratio (LCR)": "LCR", "Deposits book": "Deposits",
                           "ALM gap up to 1 month": "ALM_1m", "ALM gap up to 1 year": "ALM_1y"}),
]
baj = pd.concat(parts, axis=1).reset_index()
baj.insert(1, "Company", "Bajaj Finance")
baj.insert(2, "Basis", "Consolidated")       # Mahindra is Standalone -> kept as a column so you can state it
# ---------- Combine ----------
panel = pd.concat([mah, baj], ignore_index=True)
panel["Company"] = panel["Company"].str.strip()
panel["FY_end"] = panel["FY"].str[2:6].astype(int) + 1
panel = panel.sort_values(["Company", "FY_end"]).reset_index(drop=True)

# reported ratios come as fractions (0.029) -> convert once to percentages
frac_cols = ["GNPA_rep", "NNPA_rep", "PCR_rep", "CreditCost_rep", "ROA_rep", "ROE_rep",
             "CRAR", "Tier1", "CoF", "LCR", "ALM_1m", "ALM_1y"]
bad = panel[frac_cols].abs().drop(columns=["LCR", "ALM_1m", "ALM_1y"]).max()
print("Ratio columns already >1 before scaling (should be empty):\n", bad[bad > 1])
panel[frac_cols] = panel[frac_cols] * 100
panel = panel.rename(columns={c: c + "_pct" for c in frac_cols})

Ratio columns already >1 before scaling (should be empty):
 Series([], dtype: float64)


In [ ]:
panel = panel.sort_values(["Company", "FY_end"]).reset_index(drop=True)
log = []                                   # audit trail of every filled cell

def fill(col, values, method):
    """fill only NaN cells where `values` has a number; record each one"""
    if col not in panel: panel[col] = np.nan
    m = panel[col].isna() & values.notna()
    for i in panel.index[m]:
        log.append((panel.at[i, "Company"], panel.at[i, "FY"], col, round(float(values[i]), 2), method))
    panel.loc[m, col] = values[m]

is_b = panel["Company"].eq("Bajaj Finance")

# 1) NPA amounts <-> ratios (GNPA% = Gross NPA / loans)
base = panel["Gross_Loans"].fillna(panel["AUM"])          # Bajaj FY17 has no gross loans -> AUM
fill("Gross_NPA",    panel["GNPA_rep_pct"] / 100 * base,  "GNPA% x loan base")
fill("Net_NPA",      panel["NNPA_rep_pct"] / 100 * base,  "NNPA% x loan base")
fill("GNPA_rep_pct", panel["Gross_NPA"] / base * 100,     "Gross NPA / loan base")
fill("NNPA_rep_pct", panel["Net_NPA"] / base * 100,       "Net NPA / loan base")

# 2) Stage 3 = Gross NPA in both datasets (matches exactly FY18+). Stage 1/2 don't exist pre-Ind AS -> stay NaN.
fill("Stage3", panel["Gross_NPA"], "Stage 3 = Gross NPA")

# 3) Bajaj write-offs via ECL roll-forward: opening ECL + impairment charge - closing ECL
ecl_tot = panel[["ECL_S1", "ECL_S2", "ECL_S3"]].sum(axis=1, min_count=3)
wo_est  = ecl_tot.groupby(panel["Company"]).shift(1) + panel["Impairment"] - ecl_tot
wo_est[~is_b] = np.nan
chk = pd.DataFrame({"FY": panel["FY"], "reported": panel["Writeoffs"], "rollforward": wo_est})
print("Validation vs reported write-offs:\n", chk[is_b & chk.reported.notna() & chk.rollforward.notna()].round(0).to_string(index=False))
fill("Writeoffs", wo_est, "ECL roll-forward")

# 4) One-time provisions: blank = none disclosed -> 0 (Bajaj only)
panel.loc[is_b, "OneTime_Prov"] = panel.loc[is_b, "OneTime_Prov"].fillna(0)

# 5) Bajaj balance sheet FY21-26. ROE is on AVERAGE equity -> closing equity_t = 2*(PAT/ROE) - equity_(t-1)
#    (check: this formula gives FY20 = 32,319 vs actual 32,328)
for i in panel.index[is_b]:
    p = i - 1
    if p >= 0 and panel.at[p, "Company"] == "Bajaj Finance" and pd.isna(panel.at[i, "Net_Worth"]) and pd.notna(panel.at[p, "Net_Worth"]):
        avg_eq = panel.at[i, "PAT"] / (panel.at[i, "ROE_rep_pct"] / 100)
        panel.at[i, "Net_Worth"] = 2 * avg_eq - panel.at[p, "Net_Worth"]
        log.append(("Bajaj Finance", panel.at[i, "FY"], "Net_Worth", round(panel.at[i, "Net_Worth"]), "2 x PAT/ROE - prior equity"))

fill("Total_Assets", panel["Net_Worth"] * panel["Leverage_rep"], "Equity x reported leverage")

# borrowings FY19-21 sit in the Liquidity sheet ("Consolidated borrowings") - your read_baj never loaded it
bl = read_baj("Liquidity", {"Consolidated borrowings": "Borrowings_Liq"}).reset_index()
bl["Company"] = "Bajaj Finance"
panel = panel.merge(bl, on=["Company", "FY"], how="left")
fill("Borrowings", panel["Borrowings_Liq"], "Liquidity sheet")
panel = panel.drop(columns="Borrowings_Liq")
is_b = panel["Company"].eq("Bajaj Finance")
other = ((panel["Total_Assets"] - panel["Net_Worth"] - panel["Borrowings"]) / panel["Total_Assets"])[is_b].dropna().median()
print(f"Other liabilities = {other:.1%} of assets")
fill("Borrowings", panel["Total_Assets"] * (1 - other) - panel["Net_Worth"], "Assets x (1-other%) - equity")

# 6) LCR: interpolate only INSIDE the observed range (Bajaj FY23). Pre-Dec-2020 stays NaN (rule didn't exist)
lcr_i = panel.groupby("Company")["LCR_pct"].transform(lambda s: s.interpolate(limit_area="inside"))
fill("LCR_pct", lcr_i, "linear interpolation")

imputed = pd.DataFrame(log, columns=["Company", "FY", "Field", "Value", "Method"])
print(len(imputed), "cells filled"); print(imputed.to_string(index=False))

Validation vs reported write-offs:
        FY  reported  rollforward
FY2018-19     965.0        973.0
FY2019-20    2248.0       2263.0
FY2020-21    5554.0       5385.0
FY2021-22    4807.0       4706.0
Other liabilities = 2.4% of assets
30 cells filled
         Company        FY        Field     Value                       Method
   Bajaj Finance FY2016-17    Gross_NPA   1011.29            GNPA% x loan base
   Bajaj Finance FY2016-17      Net_NPA    264.86            NNPA% x loan base
Mahindra Finance FY2015-16 GNPA_rep_pct      8.96        Gross NPA / loan base
Mahindra Finance FY2015-16 NNPA_rep_pct      3.17          Net NPA / loan base
   Bajaj Finance FY2016-17       Stage3   1011.29          Stage 3 = Gross NPA
Mahindra Finance FY2015-16       Stage3   3420.00          Stage 3 = Gross NPA
Mahindra Finance FY2016-17       Stage3   3610.00          Stage 3 = Gross NPA
   Bajaj Finance FY2022-23    Writeoffs   3229.00             ECL roll-forward
   Bajaj Finance FY2023-24    Writeof

In [ ]:
g = panel.groupby("Company")
avg = lambda c: (panel[c] + g[c].shift(1)) / 2            # average of opening and closing

panel["AUM_growth_pct"] = (panel["AUM"] / g["AUM"].shift(1) - 1) * 100
panel["AUM_avg"] = avg("AUM")
panel["Loan_Base"] = panel["Gross_Loans"].fillna(panel["AUM"])
panel["Loan_Base_Type"] = np.where(panel["Gross_Loans"].notna(), "Gross loans", "AUM")

# asset quality
panel["GNPA_calc_pct"] = panel["Gross_NPA"] / panel["Loan_Base"] * 100
panel["NNPA_calc_pct"] = panel["Net_NPA"] / panel["Loan_Base"] * 100
panel["PCR_calc_pct"] = (1 - panel["Net_NPA"] / panel["Gross_NPA"]) * 100
panel["Stage_Total"] = panel[["Stage1", "Stage2", "Stage3"]].sum(axis=1, min_count=3)
panel["Stage2_pct"] = panel["Stage2"] / panel["Stage_Total"] * 100
panel["Stage3_pct"] = panel["Stage3"] / panel["Stage_Total"] * 100
panel["Stage2_3_pct"] = panel["Stage2_pct"] + panel["Stage3_pct"]
panel["ECL_Total"] = panel[["ECL_S1", "ECL_S2", "ECL_S3"]].sum(axis=1, min_count=3)
panel["ECL_to_Loans_pct"] = panel["ECL_Total"] / panel["Loan_Base"] * 100
panel["ECL_S3_cover_pct"] = panel["ECL_S3"] / panel["Stage3"] * 100
panel["NNPA_to_NetWorth_pct"] = panel["Net_NPA"] / panel["Net_Worth"] * 100

# credit cost / write-offs (impairment / AVERAGE AUM for both)
panel["CreditCost_calc_pct"] = panel["Impairment"] / panel["AUM_avg"] * 100
panel["Writeoff_ratio_pct"] = panel["Writeoffs"] / panel["AUM_avg"] * 100

# profitability, efficiency, leverage, liquidity
panel["ROA_calc_pct"] = panel["PAT"] / avg("Total_Assets") * 100
panel["ROE_calc_pct"] = panel["PAT"] / avg("Net_Worth") * 100
panel["ROA_final_pct"] = panel["ROA_rep_pct"].fillna(panel["ROA_calc_pct"])   # reported first, calc as fallback
panel["ROE_final_pct"] = panel["ROE_rep_pct"].fillna(panel["ROE_calc_pct"])
panel["Opex_to_NTI_pct"] = panel["Opex"] / panel["NTI"] * 100
panel["Impairment_to_NTI_pct"] = panel["Impairment"] / panel["NTI"] * 100
panel["PPOP"] = panel["NTI"] - panel["Opex"]
panel["Leverage_calc_x"] = panel["Total_Assets"] / panel["Net_Worth"]
panel["Debt_to_Equity_x"] = panel["Borrowings"] / panel["Net_Worth"]
panel["Liq_Buffer_any"] = panel["Liq_Buffer"].fillna(panel["Liq_Buffer_Supplied"])
panel["Liq_Buffer_to_AUM_pct"] = panel["Liq_Buffer_any"] / panel["AUM"] * 100

# comparability flags
panel["NPA_norm"] = np.select([panel["FY_end"] <= 2017, panel["FY_end"] <= 2021], ["4-month", "3-month"], "90dpd-strict")
panel["IndAS_stage_data"] = panel["Stage1"].notna().astype(int)
panel["Covid_year"] = panel["FY"].isin(["FY2019-20", "FY2020-21"]).astype(int)

In [ ]:
# 1) credit cost: reported first, then calc, then closing-AUM fallback for Bajaj FY17 (no FY16 opening AUM)
cc_close = panel["Impairment"] / panel["AUM"] * 100
panel["CreditCost_final_pct"] = panel["CreditCost_rep_pct"].fillna(panel["CreditCost_calc_pct"]).fillna(cc_close)
panel["Writeoff_ratio_pct"]   = panel["Writeoff_ratio_pct"].fillna(panel["Writeoffs"] / panel["AUM"] * 100)

# 2) leverage and debt/equity (D/E = Leverage x (1 - other liab%) - 1; reproduces Bajaj FY19 at 5.15 vs 5.16 actual)
panel["Leverage_final_x"]     = panel["Leverage_rep"].fillna(panel["Leverage_calc_x"])
panel["Debt_to_Equity_final"] = panel["Debt_to_Equity_x"].fillna(panel["Leverage_final_x"] * (1 - other) - 1)

# 3) Stage-3 share: Ind AS stage table if available, else Gross NPA %
panel["Stage3_pct_final"] = panel["Stage3_pct"].fillna(panel["GNPA_calc_pct"])

# 4) provisions
panel["PCR_final_pct"]       = panel["PCR_calc_pct"].fillna(panel["PCR_rep_pct"])
panel["Impairment_core"]     = panel["Impairment"] - panel["OneTime_Prov"].fillna(0)
panel["CreditCost_core_pct"] = panel["Impairment_core"] / panel["AUM_avg"] * 100

# 5) flag estimates; ROE/ROA recomputed from ESTIMATED equity would be circular, so blank them
panel["BS_estimated"] = ((panel["Company"] == "Bajaj Finance") & (panel["FY_end"] >= 2021)).astype(int)
panel.loc[panel["BS_estimated"] == 1, ["ROE_calc_pct", "ROA_calc_pct", "Leverage_calc_x"]] = np.nan

# genuine remaining gaps (columns with SOME but not ALL years blank), after the FY2016-17 window filter
final = panel[panel["FY_end"] >= 2017].reset_index(drop=True)
for name, d in final.groupby("Company"):
    n = d.isna().sum()
    print(f"\n{name} ({len(d)} yrs):"); print(n[(n > 0) & (n < len(d))].sort_values(ascending=False).to_string())


Bajaj Finance (10 yrs):
ROA_calc_pct             9
ROE_calc_pct             9
Leverage_calc_x          8
NII                      6
LCR_pct                  4
Liq_Buffer               3
Liq_Buffer_any           3
Liq_Buffer_to_AUM_pct    3
Debt_to_Equity_x         2
NNPA_to_NetWorth_pct     2
Writeoff_ratio_pct       2
Writeoffs                2
Deposits                 2
Borrowings               2
Total_Assets             2
Net_Worth                2
Gross_Loans              1
Stage1                   1
Stage2                   1
Stage_Total              1
AUM_avg                  1
AUM_growth_pct           1
AUF                      1
AUM_Standalone           1
ECL_S1                   1
ECL_S2                   1
ECL_S3                   1
CreditCost_calc_pct      1
ECL_S3_cover_pct         1
ECL_to_Loans_pct         1
ECL_Total                1
Stage3_pct               1
Stage2_3_pct             1
Stage2_pct               1
CreditCost_core_pct      1

Mahindra Finance (10 yrs):
Li

In [ ]:
# ---- Bajaj: keep real segment rows, latest classification block per year ----
sb = pd.read_excel(BAJ_PATH, sheet_name="Segment_AUM", header=3)
sb.columns = [str(c).strip() for c in sb.columns]
sb = sb[sb["Block"].astype(str).str.match(r"^[0-9A-Z]:") & sb["Segment"].notna() & (sb["Segment"] != "Total")].copy()
d = pd.to_datetime(sb["As of"], format="%d-%b-%Y")
sb["FY"] = np.where(d.dt.month == 3, "FY" + (d.dt.year - 1).astype(str) + "-" + d.dt.year.astype(str).str[-2:], "")
sb["Block_ID"] = sb["Block"].str[0]
sb = sb[sb["Block_ID"] == sb.groupby("FY")["Block_ID"].transform("max")]
seg_b = pd.DataFrame({
    "Company": "Bajaj Finance", "FY": sb["FY"], "Segment": sb["Segment"].str.strip(),
    "Segment_AUM": to_num(sb["AUM (₹ cr)"]), "Share_pct": to_num(sb["Share of AUM"]) * 100,
    "Segment_GNPA_pct": to_num(sb["Segment GNPA %"]) * 100,
    "Secured_Type": sb["Secured / Unsecured"], "Risk_Tier": sb["Risk tier (your view)"].str.replace("–", "-", regex=False),
})

# ---- Mahindra: risk tiers are YOUR judgement - edit and justify in your write-up ----
risk_map = {
    "Auto & Passenger Vehicles": ("Secured", "Low"),
    "Tractors & Farm Equipment": ("Secured", "Medium-High"),
    "Pre-Owned Vehicles": ("Secured", "High"),
    "Commercial Vehicles & CE": ("Secured", "Medium-High"),
    "SME & Enterprise Loans": ("Mostly unsecured", "High"),
    "Personal & Other Loans": ("Unsecured", "High"),
}
sm = pd.read_excel(MAH_PATH, sheet_name="Segment data")
sm.columns = sm.columns.str.strip()
sm = sm[sm["FY"].astype(str).str.match(FY_RE)]
seg_m = pd.DataFrame({
    "Company": "Mahindra Finance", "FY": sm["FY"], "Segment": sm["Segment / Product Line"].str.strip(),
    "Segment_AUM": to_num(sm["Segment_AUM_INR_Cr"]), "Share_pct": to_num(sm["Share_of_Total_AUM_Pct"]) * 100,
    "Segment_GNPA_pct": np.nan,
})
seg_m["Secured_Type"] = seg_m["Segment"].map(lambda s: risk_map.get(s, (np.nan, np.nan))[0])
seg_m["Risk_Tier"] = seg_m["Segment"].map(lambda s: risk_map.get(s, (np.nan, np.nan))[1])
print("Unmapped Mahindra segments:", seg_m.loc[seg_m["Risk_Tier"].isna(), "Segment"].unique())

seg = pd.concat([seg_b, seg_m], ignore_index=True)
seg["High_Risk"] = (seg["Risk_Tier"] == "High").astype(int)
seg["Unsecured"] = seg["Secured_Type"].str.contains("unsecured", case=False, na=False).astype(int)
seg["Segment_GNPA_amt"] = seg["Segment_AUM"] * seg["Segment_GNPA_pct"] / 100
seg["Share_recalc_pct"] = seg["Segment_AUM"] / seg.groupby(["Company", "FY"])["Segment_AUM"].transform("sum") * 100

hr = (seg.assign(h=seg["Segment_AUM"] * seg["High_Risk"], u=seg["Segment_AUM"] * seg["Unsecured"])
         .groupby(["Company", "FY"])[["h", "u", "Segment_AUM"]].sum())
hr = pd.DataFrame({"HighRisk_Share_pct": hr["h"] / hr["Segment_AUM"] * 100,
                   "Unsecured_Share_pct": hr["u"] / hr["Segment_AUM"] * 100}).reset_index()
panel = panel.merge(hr, on=["Company", "FY"], how="left")

Unmapped Mahindra segments: []


In [ ]:
w = {"Low": 0.5, "Medium": 1.0, "Medium-High": 1.5, "High": 2.0}      # YOUR assumption - state it in the report
m = seg["Company"].eq("Mahindra Finance")
seg["_score"] = seg["Segment_AUM"] * seg["Risk_Tier"].map(w)
share = seg["_score"] / seg.groupby(["Company", "FY"])["_score"].transform("sum")
gnpa_amt = seg[["Company", "FY"]].merge(panel[["Company", "FY", "Gross_NPA"]], how="left", on=["Company", "FY"])["Gross_NPA"] * share
seg.loc[m, "Segment_GNPA_amt"] = gnpa_amt[m]
seg.loc[m, "Segment_GNPA_pct"] = gnpa_amt[m] / seg.loc[m, "Segment_AUM"] * 100
seg["GNPA_is_modelled"] = m.astype(int)        # Bajaj = disclosed, Mahindra = allocated
seg = seg.drop(columns="_score")

In [ ]:
core = ["AUM", "AUM_growth_pct", "GNPA_rep_pct", "NNPA_rep_pct", "GNPA_calc_pct", "PCR_calc_pct",
        "Stage2_pct", "Stage3_pct", "CreditCost_rep_pct", "CreditCost_calc_pct", "ROA_final_pct",
        "ROE_final_pct", "CRAR_pct", "Tier1_pct", "Opex_to_NTI_pct", "Impairment_to_NTI_pct", "HighRisk_Share_pct"]

# 1) completeness (% of years filled) for the core variables
print(panel.groupby("Company")[core].apply(lambda d: d.notna().mean() * 100).T.round(0))

# 2) genuine gaps only: columns that exist for the company but have some blank years
for name, d in panel.groupby("Company"):
    d = d.drop(columns="Company")
    n = d.isna().sum()
    gaps = n[(n > 0) & (n < len(d))]
    print(f"\n{name}: columns with some blank years (out of {len(d)})")
    print(gaps.sort_values(ascending=False))

# 3) analysis-ready table: core variables only, no empty company-specific columns
analysis = panel[["Company", "FY", "Basis"] + core].copy()

Company                Bajaj Finance  Mahindra Finance
AUM                            100.0             100.0
AUM_growth_pct                  90.0              91.0
GNPA_rep_pct                   100.0             100.0
NNPA_rep_pct                   100.0             100.0
GNPA_calc_pct                  100.0             100.0
PCR_calc_pct                   100.0             100.0
Stage2_pct                      90.0              82.0
Stage3_pct                      90.0              82.0
CreditCost_rep_pct             100.0              91.0
CreditCost_calc_pct             90.0              91.0
ROA_final_pct                  100.0              91.0
ROE_final_pct                  100.0              91.0
CRAR_pct                       100.0              91.0
Tier1_pct                      100.0              91.0
Opex_to_NTI_pct                100.0             100.0
Impairment_to_NTI_pct          100.0             100.0
HighRisk_Share_pct             100.0              91.0

Bajaj Fin

In [ ]:
b = panel["Company"].eq("Bajaj Finance")
panel["NNPA_to_NetWorth_pct"] = panel["Net_NPA"] / panel["Net_Worth"] * 100
panel["Debt_to_Equity_x"]     = panel["Borrowings"] / panel["Net_Worth"]
panel["Debt_to_Equity_final"] = panel["Debt_to_Equity_x"].fillna(panel["Leverage_final_x"] * (1 - other) - 1)
panel["Leverage_final_x"]     = panel["Leverage_rep"].fillna(panel["Total_Assets"] / panel["Net_Worth"])
panel["Writeoff_ratio_pct"]   = panel["Writeoffs"] / panel["AUM_avg"] * 100
panel["Writeoff_ratio_pct"]   = panel["Writeoff_ratio_pct"].fillna(panel["Writeoffs"] / panel["AUM"] * 100)

In [ ]:
# only the variables your analysis uses -> one shared, comparable set of columns
analysis_cols = ["Company", "FY", "Basis", "AUM", "AUM_growth_pct", "GNPA_rep_pct", "NNPA_rep_pct", "PCR_final_pct",
                 "Stage2_pct", "Stage3_pct_final", "Stage2_3_pct", "ECL_to_Loans_pct", "CreditCost_final_pct",
                 "CreditCost_core_pct", "Writeoff_ratio_pct", "ROA_final_pct", "ROE_final_pct", "CRAR_pct", "Tier1_pct",
                 "Leverage_final_x", "Debt_to_Equity_final", "NNPA_to_NetWorth_pct", "Opex_to_NTI_pct",
                 "Impairment_to_NTI_pct", "LCR_pct", "Liq_Buffer_to_AUM_pct", "HighRisk_Share_pct", "Unsecured_Share_pct",
                 "BS_estimated", "Covid_year"]
final = panel[panel["FY_end"] >= 2017]
analysis = final[[c for c in analysis_cols if c in final]].reset_index(drop=True)

# gaps that remain, each with a reason
reason = {"Stage2_pct": "Pre-Ind AS: no staging in FY17", "Stage2_3_pct": "Pre-Ind AS: no staging in FY17",
          "ECL_to_Loans_pct": "Pre-Ind AS: no stage ECL in FY17", "LCR_pct": "LCR rule began Dec-2020",
          "Liq_Buffer_to_AUM_pct": "Bajaj buffer not disclosed FY17-19", "AUM_growth_pct": "No FY16 base for Bajaj",
          "Writeoff_ratio_pct": "Bajaj FY17-18 write-offs not disclosed",
          "NNPA_to_NetWorth_pct": "Bajaj FY17-18 equity unavailable", "Debt_to_Equity_final": "Bajaj FY17-18 equity unavailable"}
gaps = (analysis.drop(columns=["Basis"]).set_index(["Company", "FY"]).isna().stack().loc[lambda s: s]
        .reset_index().rename(columns={"level_2": "Variable"})[["Company", "FY", "Variable"]])
gaps["Reason"] = gaps["Variable"].map(reason).fillna("check source")
print(f"{analysis.drop(columns='Basis').notna().mean().mean():.1%} of the analysis table is filled")
print(gaps.to_string(index=False))

# Bajaj segments: keep the years that exist and use the like-for-like year range
seg_b = seg[(seg["Company"] == "Bajaj Finance") & seg["Segment_GNPA_pct"].notna()]
print("Bajaj segment GNPA exists for:", sorted(seg_b["FY"].unique()))   # FY18, FY23-FY26

96.2% of the analysis table is filled
         Company        FY              Variable                                 Reason
   Bajaj Finance FY2016-17        AUM_growth_pct                 No FY16 base for Bajaj
   Bajaj Finance FY2016-17            Stage2_pct         Pre-Ind AS: no staging in FY17
   Bajaj Finance FY2016-17          Stage2_3_pct         Pre-Ind AS: no staging in FY17
   Bajaj Finance FY2016-17      ECL_to_Loans_pct       Pre-Ind AS: no stage ECL in FY17
   Bajaj Finance FY2016-17   CreditCost_core_pct                           check source
   Bajaj Finance FY2016-17    Writeoff_ratio_pct Bajaj FY17-18 write-offs not disclosed
   Bajaj Finance FY2016-17  NNPA_to_NetWorth_pct       Bajaj FY17-18 equity unavailable
   Bajaj Finance FY2016-17               LCR_pct                LCR rule began Dec-2020
   Bajaj Finance FY2016-17 Liq_Buffer_to_AUM_pct     Bajaj buffer not disclosed FY17-19
   Bajaj Finance FY2017-18    Writeoff_ratio_pct Bajaj FY17-18 write-offs not disc

In [ ]:
print("Duplicates (Company, FY):", panel.duplicated(["Company", "FY"]).sum())
panel["chk_stage_vs_loans_pct"] = (panel["Stage_Total"] / panel["Loan_Base"] - 1) * 100
panel["chk_GNPA_gap_pp"] = panel["GNPA_calc_pct"] - panel["GNPA_rep_pct"]
panel["chk_PCR_gap_pp"] = panel["PCR_calc_pct"] - panel["ECL_S3_cover_pct"]

print("\nStage total vs loan base differs >2%:")
print(panel.loc[panel["chk_stage_vs_loans_pct"].abs() > 2, ["Company", "FY", "Stage_Total", "Loan_Base"]])
print("\nCalc GNPA differs >0.5pp from reported:")
print(panel.loc[panel["chk_GNPA_gap_pp"].abs() > 0.5, ["Company", "FY", "GNPA_rep_pct", "GNPA_calc_pct"]])
print("\nPCR vs Stage-3 ECL cover differs >2pp:")
print(panel.loc[panel["chk_PCR_gap_pp"].abs() > 2, ["Company", "FY", "PCR_calc_pct", "ECL_S3_cover_pct"]])

# study window FY2016-17 .. FY2025-26 (FY2015-16 was only needed for averages / growth)
panel = panel[panel["FY_end"] >= 2017].reset_index(drop=True)
seg = seg[seg["FY"].isin(panel["FY"].unique())].reset_index(drop=True)

miss = panel.isna().groupby(panel["Company"]).sum().T
print("\nMissing values:\n", miss[(miss > 0).any(axis=1)])
print(panel.shape, seg.shape)

# export + download
with pd.ExcelWriter("NBFC_master_dataset.xlsx") as w:
    panel.to_excel(w, "Master_Panel", index=False)
    seg.to_excel(w, "Segments", index=False)
panel.to_csv("NBFC_master_panel.csv", index=False)
files.download("NBFC_master_dataset.xlsx")

# handy wide view for side-by-side comparison
panel.pivot(index="FY", columns="Company", values=["GNPA_rep_pct", "PCR_calc_pct", "CreditCost_calc_pct", "ROA_final_pct", "HighRisk_Share_pct"])

Duplicates (Company, FY): 0

Stage total vs loan base differs >2%:
             Company         FY  Stage_Total  Loan_Base
12  Mahindra Finance  FY2017-18      45837.0    48529.0
14  Mahindra Finance  FY2019-20      77036.0    68352.0
15  Mahindra Finance  FY2020-21      69700.0    64610.0
16  Mahindra Finance  FY2021-22      67160.0    65830.0
19  Mahindra Finance  FY2024-25     118500.0   124170.0
20  Mahindra Finance  FY2025-26     139310.0   145100.0

Calc GNPA differs >0.5pp from reported:
             Company         FY  GNPA_rep_pct  GNPA_calc_pct
15  Mahindra Finance  FY2020-21           9.0       9.719858

PCR vs Stage-3 ECL cover differs >2pp:
Empty DataFrame
Columns: [Company, FY, PCR_calc_pct, ECL_S3_cover_pct]
Index: []

Missing values:
 Company                 Bajaj Finance  Mahindra Finance
Stage1                              1                 1
Stage2                              1                 1
Gross_Loans                         1                 0
Total_Assets   

/tmp/ipykernel_2870/1142886169.py:23: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  panel.to_excel(w, "Master_Panel", index=False)
/tmp/ipykernel_2870/1142886169.py:24: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  seg.to_excel(w, "Segments", index=False)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

GNPA_rep_pct                   PCR_calc_pct                  CreditCost_calc_pct                  ROA_final_pct                  HighRisk_Share_pct                 
Company   Bajaj Finance Mahindra Finance Bajaj Finance Mahindra Finance       Bajaj Finance Mahindra Finance Bajaj Finance Mahindra Finance      Bajaj Finance Mahindra Finance
FY                                                                                                                                                                             
FY2016-17          1.68              9.0     73.809524        60.886427                 NaN         2.202962          3.70              1.0          65.007642        22.999341
FY2017-18          1.41              8.5     67.500000        43.151515            1.444418         3.418563          3.70              1.8          65.780110        24.000262
FY2018-19          1.54              5.5     59.437086        38.533058            1.513792         1.833237          4.20              2.6          47.967003        26.999911
FY2019-20          1.61              8.4     60.304697        30.856248            2.987367         2.850521          4.15              1.3          50.579329        28.001194
FY2020-21          1.79              9.0     58.403515        37.579618            3.978007         4.902675          3.07              0.5          48.775066        28.000106
FY2021-22          1.60              7.7     58.027450        56.136543            2.741446         3.106613          4.16              1.3          45.198833        30.000000
FY2022-23          0.94              4.5     63.813230        59.183673            1.434253         1.152402          5.30              2.3          44.767745        31.999543
FY2023-24          0.85              3.4     57.031250        62.500000            1.602439         1.256529          5.08              1.6          42.923642        33.000460
FY2024-25          0.96              3.1     53.720050        65.714286            2.132010         1.104875          4.60              1.8          40.453030        33.000856
FY2025-26          1.01              2.9     59.718695        66.983373            2.219210         0.987241          4.30              1.9          36.820432        33.000000